# Exp 36: Model comparison

Card: [exp36_model_comparison.md](../../docs/neuroslope/experiments/exp36_model_comparison.md)


In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

from training.config import ROOT
from training.config import VIP
from training.cv import CVResult
from analysis.gradcam import GradCamFigures
from analysis.plots import pr_curves, save_figures, use_style
from analysis.thresholds import best_f1
from training.config import CLASSES

N_SAMPLE = 16
SEED = 0
CONFIG_DIR = ROOT / "configs" / "models"
MODELS = {
    "exp26 ResNet-18": CONFIG_DIR / "exp26_coarse_dropout_5fold" / "resnet18.yaml",
    "EfficientNet-B0": CONFIG_DIR / "exp32_efficientnet_b0_5fold" / "efficientnet_b0.yaml",
    "RegNetY-016": CONFIG_DIR / "exp33_regnety_016" / "regnety_016.yaml",
    "ConvNeXt-Tiny": CONFIG_DIR / "exp34_convnext_tiny" / "convnext_tiny.yaml",
}
TIMM_RESULTS = "https://raw.githubusercontent.com/huggingface/pytorch-image-models/main/results/results-imagenet.csv"
FIGURES = ROOT / "docs" / "figures"
PREFIX = "exp36_model_comparison"

# dataviz reference palette, light mode; exp26 keeps orange from earlier cards
SERIES = dict(zip(MODELS, ["#eb6834", "#2a78d6", "#1baf7a", "#eda100"]))
use_style()

## ImageNet scores


In [ ]:
# timm evaluation at 224 px, our training size
imagenet = pd.read_csv(TIMM_RESULTS)
weights = {c.timm_model: name for name, c in MODELS.items()}
imagenet = (
    imagenet[imagenet["model"].isin(weights) & (imagenet["img_size"] == 224)]
    .assign(name=lambda d: d["model"].map(weights))
    .set_index("name")
    .loc[list(MODELS), ["model", "top1", "top5"]]
)
imagenet

## Precision-recall curves


In [ ]:
results = {name: CVResult.load(c) for name, c in MODELS.items()}
fig = pr_curves(results, colors=SERIES, title="Out-of-fold rows of folds 0–4")
figures = {f"{PREFIX}_pr_curves.png": fig}
plt.show()

## Thresholds

In [ ]:
# best-F1 threshold per model and class on pooled OOF rows, as in submission
thresholds = pd.concat({name: best_f1(r.labels, r.probs, CLASSES)[["threshold", "f1"]] for name, r in results.items()}, axis=1)
thresholds.loc["Macro F1"] = [thresholds[col].mean() if col[1] == "f1" else float("nan") for col in thresholds.columns]
thresholds.round(3)

## Grad-CAM VIP: four models


In [ ]:
cams = {name: GradCamFigures(c) for name, c in MODELS.items()}
for cls in VIP:
    rows = cams["exp26 ResNet-18"].positive_rows(cls, N_SAMPLE, SEED)
    figures[f"{PREFIX}_grad_cam_{cls.lower()}_compare.png"] = GradCamFigures.compare(cams, cls, rows)
plt.show()

In [ ]:
save_figures(figures, FIGURES)